# Phase 7 (Prior Case Retrieval) — Run #5: Phase-1 **Decoder-only model 01** (12-layer MHA, NoPE) as a bi-encoder, with **QLoRA** (hand-written NF4 4-bit base + LoRA)

Same pipeline as runs #1–#4, and the **same backbone as run #3** (full fine-tuning): runs #3 and #5 differ only in the adaptation, so their gap is the price of *freezing a 4-bit copy of the model and learning a low-rank correction* versus updating every weight. Edit **Cell 2** only.

### QLoRA (Dettmers et al. 2023)
Every `nn.Linear` inside the 12 blocks (`q, k, v, o, gate, up, down`: 84 matrices, ~38M weights) is replaced by a **frozen NF4 copy** plus a trainable LoRA branch:

`y = dequant(Q(W)) x + (α/r) · B A x`,  `A ∈ R^{r×d_in}` (Kaiming-uniform), `B ∈ R^{d_out×r}` (zeros), r = 16, α = 32.

`B = 0` makes step 0 exactly the **quantised** Phase-1 model. The embedding (tied with the unused LM head) and the RMSNorm gains stay in full precision and frozen, as in the paper.

### NF4, written out (Cell 15)
1. **Blockwise absmax.** Flatten `W`, cut it into blocks of 64, divide each block by `s_b = max|w|` → values in `[−1, 1]`. Small blocks keep one outlier from crushing the resolution of a whole row.
2. **NormalFloat levels.** Trained weights are close to `N(0, σ²)`, so the best 16 levels for them are the **quantiles** of a normal distribution: each level then receives the same share of weights, which maximises the information carried per 4-bit code. Exactly as in the paper, 8 positive levels are `Φ⁻¹` of 9 equally spaced probabilities from 0.5 to `δ = 0.9677` (the last dropped), 7 negative levels the mirror image of 8 such probabilities, plus an **exact zero**; all divided by the largest. The cell recomputes them this way and checks them against the published table.
3. **Encode** every value as the index of its nearest level (4 bits), **pack two indices per byte**.
4. **Double quantisation.** The block scales themselves (one fp32 per 64 weights = 0.5 bit/weight) are quantised to 8 bits in blocks of 256 after subtracting their mean, leaving ≈ 0.127 bit/weight of scale overhead.
5. **Forward.** Unpack → look up the levels → multiply by the de-quantised scales → `F.linear` in fp16. Gradients never touch the 4-bit codes; they flow only into `A`, `B` and the head.

The cell reports the **quantisation error** (relative Frobenius error per matrix), the **LM loss on a legal sentence** for the full-precision and the NF4 model, the weight memory in both formats, and the **zero-shot dense retrieval of the 4-bit model** — to be compared with run #3's full-precision zero-shot table. Honest caveat for the report: with a 57M model, memory is never the bottleneck here; the run measures the *accuracy* cost of 4-bit weights and of a low-rank update, which is the scientific content of QLoRA.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy, hashlib, bisect, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu130


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p7_dec01_qlora"
METHOD     = "qlora"          # Phase-7 list: QLoRA
SMOKE_TEST = False            # True: SMOKE_STEPS steps, dev on SMOKE_DEV_Q queries (full dev pool), NO test / export
ATTN_MODE  = "causal"         # the NoPE checkpoint infers position from the causal mask: causal is the only legal mode

# ---- Phase-1 checkpoint (same file as Phases 2-6) --------------------------------------------------
CKPT_KIND = "legacy_nope"     # the original model-01 checkpoint, loaded as NoPE (what was actually trained)
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-01-best/best.pt") if IS_KAGGLE else Path("./input/best_dec01.pt")

# ---- Data + audit artifacts (same as runs #1-#2) ---------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"
LOCAL_DATA_DIR = None
P7    = "Phase_7_PCR"
FILES = {"train": ("Train/train.json", "Train/train_candidates.json"),
         "dev":   ("Dev/dev.json",     "Dev/dev_candidates.json"),
         "test":  ("Test/test.json",   "Test/test_candidates.json")}
BM25_PATH   = Path("/kaggle/input/datasets/ascyrax/caps-p7-bm25/bm25_top300.json") if IS_KAGGLE else Path("./input/bm25_top300.json")
TOKCACHE_IN = Path("/kaggle/input/datasets/ascyrax/caps-p7-tokcache") if IS_KAGGLE else Path("./input/tokcache")
N_PROC      = 4

# ---- Views (identical to runs #1-#2) ---------------------------------------------------------------
CHUNK_TOKENS   = 512          # [BOS] + 510 text tokens + [EOS]
MAX_DOC_CHUNKS = 16
QUERY_VIEW     = "ctx+head"
CTX_WINDOW     = 1
Q_MAX_CHUNKS   = 4
CIT_TAG        = "<CITATION>"

# ---- Encoder / scoring ----------------------------------------------------------------------------
POOLING    = "auto"           # "mean" | "wmean" | "last" | "auto"
SCORE_MODE = "auto"           # "mean" | "maxsim" | "auto"
USE_HEAD   = True
TAU        = 0.05
EMB_BATCH  = 64

# ---- Training -------------------------------------------------------------------------------------
EPOCHS        = 4
BATCH_PAIRS   = 32
C_TRAIN       = 4
QC_TRAIN      = 2
HN_TOP        = 50
FWD_MICRO     = 32
WARMUP_FRAC   = 0.06
GRAD_CLIP     = 1.0
GRAD_CKPT     = True
TIME_BUDGET_MIN = 360
HP = {"qlora": dict(lr=3e-4, head_lr=2.5e-4, wd=0.0, r=16, alpha=32, dropout=0.05)}
QLORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
NF4_BLOCK, DQ_BLOCK, DOUBLE_QUANT = 64, 256, True
POS_SAMPLING  = "uniform"     # "uniform" | "bm25_hard" (see run #3)
HARD_RANK, HARD_BOOST = 50, 2.0

# ---- Fusion / evaluation --------------------------------------------------------------------------
K_EVAL     = 7
ALPHA_GRID = tuple(np.round(np.arange(0.0, 1.0001, 0.05), 2))
RRF_K      = 60
RECALL_KS  = (7, 20, 50, 100, 300)

# ---- Exports --------------------------------------------------------------------------------------
EXPORT_TEACHER = True
SAVE_SCORES    = True
SAVE_TOP       = 300

# ---- Smoke test -----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_DEV_Q = 40, 40

# ---- derived --------------------------------------------------------------------------------------
assert METHOD in HP and POOLING in ("mean", "wmean", "last", "auto") and SCORE_MODE in ("mean", "maxsim", "auto")
assert QUERY_VIEW in ("ctx", "head", "ctx+head") and POS_SAMPLING in ("uniform", "bm25_hard") and ATTN_MODE == "causal"
BODY = CHUNK_TOKENS - 2
MAX_DOC_TOK = MAX_DOC_CHUNKS * BODY
MAX_DOC_CHARS = int(MAX_DOC_TOK * 4.6)
RUN_DIR = WORK_DIR / "runs" / RUN_NAME; RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | {METHOD} {HP[METHOD]} | NF4 block {NF4_BLOCK} double-quant {DOUBLE_QUANT} | ckpt {CKPT_KIND} | chunks {CHUNK_TOKENS} x <= {MAX_DOC_CHUNKS} | query view {QUERY_VIEW} | "
      f"B={BATCH_PAIRS} pairs, C={C_TRAIN}, QC={QC_TRAIN}, tau={TAU} | epochs {EPOCHS} | pos sampling {POS_SAMPLING} | smoke={SMOKE_TEST}")

RUN p7_dec01_qlora | qlora {'lr': 0.0003, 'head_lr': 0.00025, 'wd': 0.0, 'r': 16, 'alpha': 32, 'dropout': 0.05} | NF4 block 64 double-quant True | ckpt legacy_nope | chunks 512 x <= 16 | query view ctx+head | B=32 pairs, C=4, QC=2, tau=0.05 | epochs 4 | pos sampling uniform | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_01.ipynb` (Cells 4-10)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model configuration (decoder-only model 01, corrected)
#
#   preset_rope_mha()  -> model 01 AS INTENDED: RoPE + MHA + RMSNorm pre-norm + SwiGLU (retrain)
#   preset_nope_mha()  -> model 01 AS ACTUALLY TRAINED: identical but with no positional encoding.
#                         The original notebook built RoPE but never passed it to attention.
#                         Used to load the old checkpoint faithfully (MODE = "reuse", Cell 16).

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 12
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # == n_head -> MHA; < n_head -> GQA
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"            # "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "attention"
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("rope", "learned", "none")
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, pos_type="rope", max_seq_len=max_seq_len)

def preset_nope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, pos_type="none", max_seq_len=max_seq_len)

PRESETS = {"rope_mha": preset_rope_mha, "nope_mha": preset_nope_mha}
print(json.dumps(asdict(preset_rope_mha()), indent=2))

{
  "vocab_size": 32000,
  "n_layer": 12,
  "d_model": 512,
  "n_head": 8,
  "n_kv_head": 8,
  "head_dim": 64,
  "ff_mult": 2.6666666666666665,
  "d_ff": 1408,
  "max_seq_len": 1024,
  "norm_type": "rmsnorm",
  "norm_eps": 1e-05,
  "pos_type": "rope",
  "rope_theta": 10000.0,
  "mixer_type": "attention",
  "mlp_type": "swiglu",
  "channel_mixer_type": "mlp",
  "bias": false,
  "attn_dropout": 0.0,
  "resid_dropout": 0.0,
  "embd_dropout": 0.0,
  "tie_embeddings": true,
  "init_std": 0.02,
  "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Rotary position embeddings (RoPE)
#
# RoPE rotates each (even, odd) pair of q and k dimensions by angle p * theta^(-2i/D) at position p.
# Because rotations compose, q_i^T k_j depends only on the offset (i - j): relative position
# enters the attention score directly, with no parameters.
#
# FIX: in the original notebook this module was built but never reached the attention layers.
# Here the Backbone computes (cos, sin) once per forward and passes them to every layer.

class RoPE(nn.Module):
    """Table built for 2x the training length so generation never overflows it."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [8]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [10]:
# Cell 9 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        q_pos = torch.arange(past, S, device=device)
        k_pos = torch.arange(S, device=device)
        bias = torch.zeros(T, S, device=device).masked_fill(
            k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]          # causal
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [11]:
# Cell 10 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})


def load_legacy_model01(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    """Load a checkpoint written by the ORIGINAL decoder-only notebook.
    Parameter names are unchanged, so the weights load strictly. The config is rebuilt with
    pos_type="none" because that is what was actually trained (RoPE was never applied).
    Loading it as "rope" would silently change the function the weights compute."""
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    old = ckpt["config"]
    kept = {k: v for k, v in old.items() if k in ModelConfig.__dataclass_fields__}
    dropped = sorted(set(old) - set(kept))
    kept["pos_type"] = "none"
    cfg = ModelConfig(**kept)
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"], strict=True)
    print(f"legacy model 01 loaded as NoPE | dropped old config fields: {dropped}")
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [12]:
# Cell 11 — Load the Phase-1 checkpoint and sanity-check it
if CKPT_KIND == "legacy_nope":
    lm, lm_cfg, lm_extra = load_legacy_model01(CKPT_PATH, map_location="cpu")    # loads as NoPE, as it was trained
else:
    lm, lm_cfg, lm_extra = load_model(CKPT_PATH, map_location="cpu")
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} | layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} | "
      f"params total={pc['total']/1e6:.2f}M (embedding {pc['embedding']/1e6:.2f}M)")
if lm_extra.get("val_loss") is not None:
    print("logged Phase-1 val loss:", lm_extra["val_loss"])

# If the weights did not load properly, the LM loss on a normal legal sentence would sit near ln(V) ~ 10.4.
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"

if ATTN_MODE == "bidir" and lm_cfg.pos_type == "none":
    raise RuntimeError("ATTN_MODE='bidir' removes the causal mask, but this checkpoint has NO positional encoding "
                       "(it infers position from the mask). Use ATTN_MODE='causal', or a RoPE checkpoint (CKPT_KIND='new').")

legacy model 01 loaded as NoPE | dropped old config fields: ['causal', 'grad_checkpointing']
Phase-1 model | pos_type=none | layers=12 d_model=512 | params total=54.93M (embedding 16.38M)
logged Phase-1 val loss: 2.116826470941305
LM loss on a legal sentence: 2.015  (ln V = 10.37)


## Phase-7 data (the audit's loader) and the BM25 lists
The loader is the audit's (Cell 4), unchanged. IDs are matched on a canonical key (leading zeros stripped); every output uses the pool's own ID string. The BM25 lists come from the audit's Cell 12 (`bm25_top300.json`, best configuration, no year filter).

In [13]:
# Cell 13 — Load the 6 files (audit loader) + BM25 lists
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    rel = f"{P7}/{rel}"
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P7} and set LOCAL_DATA_DIR.") from ex

def read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

ID_KEYS   = ("id", "query_id", "case_id", "candidate_id", "doc_id", "document_id", "qid", "cid")
TEXT_KEYS = ("text", "case_text", "document", "content", "sentences", "judgment", "judgement", "body")
GOLD_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "positives", "labels", "gold", "citations", "cited")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def as_sents(v):
    if isinstance(v, str): return [s for s in v.split("\n")]
    out = []
    for x in v:
        if isinstance(x, list): out.extend(str(y) for y in x)
        elif isinstance(x, dict): out.append(str(_first(x, TEXT_KEYS + ("sentence",), "")))
        else: out.append(str(x))
    return out
def canon(x): s = str(x).strip(); return s.lstrip("0") or "0"
def unwrap(raw):
    if isinstance(raw, dict) and len(raw) == 1:
        v = next(iter(raw.values()))
        if isinstance(v, (list, dict)) and len(v) > 1: return v
    if isinstance(raw, dict) and isinstance(raw.get("data"), list): return raw["data"]
    return raw
def items_of(raw):
    raw = unwrap(raw)
    if isinstance(raw, dict):
        out = []
        for k, v in raw.items():
            if isinstance(v, dict): v = dict(v); v.setdefault("id", k)
            else: v = {"id": k, "text": v}
            out.append(v)
        return out
    return list(raw)

DOCS = {}                      # canonical id -> {"id": original string, "sents": [...]}, shared by all splits (pools overlap)
POOL, QRY = {}, {}             # split -> list of canonical ids (pool order) ; split -> list of query dicts
for split, (qf, cf) in FILES.items():
    raw_c = read_json_any(fetch(cf)); order, seen = [], set()
    for e in items_of(raw_c):
        k = canon(_first(e, ID_KEYS))
        if k in seen: continue
        seen.add(k); order.append(k)
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS)), sents=as_sents(_first(e, TEXT_KEYS)))
    POOL[split] = order; del raw_c
    raw_q = read_json_any(fetch(qf)); qs = []
    pool_set = set(order)
    for i, e in enumerate(items_of(raw_q)):
        k = canon(_first(e, ID_KEYS, i))
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS, i)), sents=as_sents(_first(e, TEXT_KEYS)))
        g = _first(e, GOLD_KEYS)
        gold = None
        if split != "test":
            g = [g] if isinstance(g, (str, int)) else (g or [])
            gold = []
            for x in g:
                c = canon(x)
                if c in pool_set and c not in gold: gold.append(c)
        qs.append(dict(key=k, id=str(_first(e, ID_KEYS, i)), gold=gold))
    QRY[split] = qs; del raw_q; gc.collect()
    print(f"{split:5s}: {len(qs)} queries | pool {len(order)} | queries inside their own pool: {sum(q['key'] in pool_set for q in qs)}"
          + (f" | gold pairs {sum(len(q['gold']) for q in qs)} | queries without gold {sum(not q['gold'] for q in qs)}" if split != "test" else ""))
print("distinct documents over all splits:", len(DOCS))

_bm = json.loads(BM25_PATH.read_text())
BM25_CFG, BM25 = _bm["config"], {s: {canon(q): [(canon(c), float(v)) for c, v in lst] for q, lst in d.items()} for s, d in _bm["lists"].items()}
del _bm
print("BM25 config:", BM25_CFG)
for s in FILES:
    miss = sum(q["key"] not in BM25[s] for q in QRY[s]); assert miss == 0, f"{s}: {miss} queries have no BM25 list"

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train: 827 queries | pool 4320 | queries inside their own pool: 827 | gold pairs 5308 | queries without gold 10


Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

... (3 outputs left out)

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

test : 237 queries | pool 1727 | queries inside their own pool: 237
distinct documents over all splits: 5825
BM25 config: {'ng': 'bi', 'k1': 8.0, 'b': 1.0, 'mode': 'binary', 'H@7': 395, 'F1@7': 0.43913285158421345, 'recall@100': 0.7543679342240494, 'MRR': 0.8950390337803802, 'year_slack': None}


## Tokenisation, views, cache
Every distinct document is tokenised once (pools overlap) into its **document view** (head 2/3 + tail 1/3 of `MAX_DOC_CHARS` characters, then at most `MAX_DOC_CHUNKS × 510` tokens), and every query additionally into its **evidence view** (the `<CITATION>` sentences ± `CTX_WINDOW`, in document order, at most `Q_MAX_CHUNKS × 510` tokens). The cache is keyed by the view settings and written to `WORK_DIR/tokcache/`. **Upload that folder as the Kaggle dataset `caps-p7-tokcache`**: every later Phase-7 run uses the same tokenizer and reads it instead of re-tokenising (~5–15 min saved per run).

In [14]:
# Cell 14 — Tokenise (multiprocessing) + cache; chunking
def cut_text(t):
    if len(t) <= MAX_DOC_CHARS: return t
    h = 2 * MAX_DOC_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_DOC_CHARS - h):]
def ctx_text(sents):
    hit = [j for j, s in enumerate(sents) if CIT_TAG in s]
    keep = sorted({k for j in hit for k in range(max(0, j - CTX_WINDOW), min(len(sents), j + CTX_WINDOW + 1))})
    return "\n".join(sents[k] for k in keep)

_W_ITEMS = None
def _tok_work(i):
    k, want_ctx = _W_ITEMS[i]; sents = DOCS[k]["sents"]
    doc = np.asarray(tokenizer.encode(cut_text("\n".join(sents)))[:MAX_DOC_TOK], dtype=np.uint16)
    ctx = None
    if want_ctx:
        t = ctx_text(sents)
        ctx = np.asarray(tokenizer.encode(t[: int(Q_MAX_CHUNKS * BODY * 4.6)])[: Q_MAX_CHUNKS * BODY], dtype=np.uint16) if t else np.zeros(0, np.uint16)
    return k, doc, ctx

SIG = hashlib.md5(json.dumps([CHUNK_TOKENS, MAX_DOC_CHUNKS, CTX_WINDOW, Q_MAX_CHUNKS, CIT_TAG, VOCAB_SIZE]).encode()).hexdigest()[:10]
CACHE_NAME = f"p7_tok_{SIG}.pkl"
cache_out = WORK_DIR / "tokcache" / CACHE_NAME; cache_out.parent.mkdir(parents=True, exist_ok=True)
TOKD, TOKQ = {}, {}
for p in (TOKCACHE_IN / CACHE_NAME, cache_out):
    if p.exists():
        TOKD, TOKQ = pickle.loads(p.read_bytes()); print(f"token cache loaded: {p} ({len(TOKD)} docs, {len(TOKQ)} query views)"); break
qkeys = {q["key"] for s in FILES for q in QRY[s]}
todo = [(k, k in qkeys) for k in DOCS if k not in TOKD or (k in qkeys and k not in TOKQ)]
if todo:
    t0 = time.time(); _W_ITEMS = todo
    with mp.get_context("fork").Pool(N_PROC) as pool:
        for k, d, c in tqdm(pool.imap_unordered(_tok_work, range(len(todo)), chunksize=4), total=len(todo), desc="tokenise"):
            TOKD[k] = d
            if c is not None: TOKQ[k] = c
    cache_out.write_bytes(pickle.dumps((TOKD, TOKQ), protocol=4))
    print(f"tokenised {len(todo)} documents in {time.time() - t0:.0f}s -> {cache_out} ({cache_out.stat().st_size / 1e6:.0f} MB)")

def chunks_of(ids, max_chunks):
    ids = ids[: max_chunks * BODY]
    ch = [ids[i:i + BODY] for i in range(0, len(ids), BODY)]
    if len(ch) > 1 and len(ch[-1]) < 32: ch = ch[:-1]          # a 5-token tail chunk is noise
    return ch or [np.zeros(0, np.uint16)]
def doc_chunks(k): return chunks_of(TOKD[k], MAX_DOC_CHUNKS)
def query_chunks(k):
    head = chunks_of(TOKD[k], 1)
    ctx = chunks_of(TOKQ[k], Q_MAX_CHUNKS) if len(TOKQ.get(k, ())) else []
    if QUERY_VIEW == "head": return chunks_of(TOKD[k], Q_MAX_CHUNKS)
    if QUERY_VIEW == "ctx": return ctx or head
    return (head + ctx)[:Q_MAX_CHUNKS]
_nq = [len(query_chunks(q["key"])) for q in QRY["train"]]; _nd = [len(doc_chunks(k)) for k in POOL["train"]]
print(f"train: query chunks/query mean {np.mean(_nq):.2f} | doc chunks/doc mean {np.mean(_nd):.2f} (cap {MAX_DOC_CHUNKS}) | "
      f"queries with an empty ctx view: {sum(len(TOKQ.get(q['key'], ())) == 0 for q in QRY['train'])}")

token cache loaded: /kaggle/input/datasets/ascyrax/caps-p7-tokcache/p7_tok_18962ef646.pkl (5825 docs, 1182 query views)
train: query chunks/query mean 3.70 | doc chunks/doc mean 9.98 (cap 16) | queries with an empty ctx view: 1


## The adapted model, embedding, scoring, evaluation, fusion

In [15]:
# Cell 15 — QLoRA: hand-written NF4 (blockwise absmax, NormalFloat levels, double quantisation) + LoRA; bi-encoder construction
from statistics import NormalDist

def nf4_levels(delta=0.9677083):
    nd = NormalDist()
    pos = [nd.inv_cdf(p) for p in np.linspace(delta, 0.5, 9)[:-1]]        # 8 positive levels
    neg = [-nd.inv_cdf(p) for p in np.linspace(delta, 0.5, 8)[:-1]]       # 7 negative levels
    v = np.sort(np.asarray(pos + [0.0] + neg)); return torch.tensor(v / np.abs(v).max(), dtype=torch.float32)
NF4 = nf4_levels()
_NF4_REF = [-1.0, -0.6961928, -0.5250731, -0.3949175, -0.2844414, -0.1847734, -0.0910500, 0.0,
            0.0795803, 0.1609302, 0.2461123, 0.3379152, 0.4407098, 0.5626170, 0.7229568, 1.0]
assert torch.allclose(NF4, torch.tensor(_NF4_REF), atol=1e-6), "NF4 levels differ from the published table"
print("NF4 levels:", [round(x, 4) for x in NF4.tolist()])

def nf4_quantize(W):
    # W [out, in] -> dict of packed 4-bit codes + (double-quantised) block scales
    w = W.detach().float().flatten(); n = w.numel(); pad = (-n) % NF4_BLOCK
    w = F.pad(w, (0, pad)).view(-1, NF4_BLOCK)
    s = w.abs().amax(1).clamp(min=1e-12)                                   # absmax per 64-block
    x = w / s[:, None]
    codes = (x.unsqueeze(-1) - NF4.to(x.device)).abs().argmin(-1).to(torch.uint8).flatten()   # nearest level
    if codes.numel() % 2: codes = F.pad(codes, (0, 1))
    packed = codes[0::2] | (codes[1::2] << 4)                              # two 4-bit codes per byte
    q = dict(packed=packed, shape=tuple(W.shape), n=n)
    if DOUBLE_QUANT:                                                       # 8-bit scales in blocks of 256, after removing the mean
        m = s.mean(); sc = F.pad(s - m, (0, (-s.numel()) % DQ_BLOCK)).view(-1, DQ_BLOCK)
        c2 = sc.abs().amax(1).clamp(min=1e-12) / 127.0
        q.update(qs=torch.round(sc / c2[:, None]).clamp(-127, 127).to(torch.int8), c2=c2, mean=m, nb=s.numel())
    else:
        q.update(s=s)
    return q

def nf4_dequantize(q, levels, dtype=torch.float32):
    p = q["packed"]
    codes = torch.stack([p & 0x0F, p >> 4], 1).flatten().long()
    if DOUBLE_QUANT: s = (q["qs"].float() * q["c2"][:, None]).flatten()[: q["nb"]] + q["mean"]
    else: s = q["s"]
    w = (levels[codes[: s.numel() * NF4_BLOCK]].view(-1, NF4_BLOCK) * s[:, None]).flatten()[: q["n"]]
    return w.view(q["shape"]).to(dtype)

class QLoRALinear(nn.Module):
    # y = dequant(NF4(W)) x + (alpha/r) B A dropout(x); the 4-bit base is a set of buffers (never trained)
    def __init__(self, base: nn.Linear, r, alpha, dropout):
        super().__init__()
        q = nf4_quantize(base.weight)
        self.shape, self.n = q["shape"], q["n"]
        self.register_buffer("packed", q["packed"])
        self.register_buffer("levels", NF4.clone())
        if DOUBLE_QUANT:
            self.register_buffer("qs", q["qs"]); self.register_buffer("c2", q["c2"]); self.register_buffer("mean", q["mean"].reshape(()))
            self.nb = q["nb"]
        else:
            self.register_buffer("s", q["s"])
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)
        self.in_features, self.out_features = base.in_features, base.out_features
    def _q(self):
        d = dict(packed=self.packed, shape=self.shape, n=self.n)
        if DOUBLE_QUANT: d.update(qs=self.qs, c2=self.c2, mean=self.mean, nb=self.nb)
        else: d.update(s=self.s)
        return d
    def weight_fp(self, dtype=torch.float32):
        return nf4_dequantize(self._q(), self.levels, dtype)
    def forward(self, x):
        W = self.weight_fp(x.dtype if x.dtype in (torch.float16, torch.bfloat16) else torch.float32)
        y = F.linear(x, W, None if self.bias is None else self.bias.to(W.dtype))
        return y + (self.drop(x) @ self.A.t().to(x.dtype) @ self.B.t().to(x.dtype)) * self.scale
    def bytes_4bit(self):
        b = self.packed.numel() + (self.qs.numel() + 4 * self.c2.numel() + 4 if DOUBLE_QUANT else 4 * self.s.numel())
        return b

def apply_adapter(module, targets, make):
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n

def lm_loss_on(model_lm):
    _sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
    _ids = [BOS_ID] + tokenizer.encode(_sent)
    _x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        _, l, _ = model_lm(_x, targets=_y)
    return l.item()

def build_model():
    # Fresh copy of the Phase-1 weights -> every target Linear becomes a frozen NF4 base + LoRA; head identity-initialised.
    hp = HP[METHOD]
    model = copy.deepcopy(lm).cpu()
    for p in model.parameters(): p.requires_grad = False
    fp_layers = {n: m.weight.detach().clone() for n, m in model.backbone.named_modules()
                 if isinstance(m, nn.Linear) and n.split(".")[-1] in QLORA_TARGETS}
    t0 = time.time()
    n = apply_adapter(model.backbone, QLORA_TARGETS, lambda l: QLoRALinear(l, hp["r"], hp["alpha"], hp["dropout"]))
    errs, b4, bfp = [], 0, 0
    for name, m in model.backbone.named_modules():
        if isinstance(m, QLoRALinear):
            W = fp_layers[name]; Wq = m.weight_fp()
            errs.append(((W - Wq).norm() / W.norm()).item()); b4 += m.bytes_4bit(); bfp += 2 * W.numel()
    QSTATS.update(n_layers=n, rel_err_mean=float(np.mean(errs)), rel_err_max=float(np.max(errs)), mb_fp16=bfp / 2**20, mb_nf4=b4 / 2**20,
                  bits_per_weight=8 * b4 / (bfp / 2), seconds=time.time() - t0)
    print(f"NF4: {n} Linear layers quantised in {QSTATS['seconds']:.0f}s | relative error ||W-Q(W)||/||W||: mean {QSTATS['rel_err_mean']:.4f} "
          f"max {QSTATS['rel_err_max']:.4f} | weights {QSTATS['mb_fp16']:.1f} MB (fp16) -> {QSTATS['mb_nf4']:.1f} MB "
          f"({QSTATS['bits_per_weight']:.3f} bits/weight incl. scales)")
    model.to(device)
    QSTATS["lm_loss_fp"] = lm_loss_on(lm); QSTATS["lm_loss_nf4"] = lm_loss_on(model.eval())
    print(f"LM loss on a legal sentence: full precision {QSTATS['lm_loss_fp']:.4f} | NF4 base {QSTATS['lm_loss_nf4']:.4f}")
    with torch.no_grad():                                         # B = 0: the adapted model IS the quantised model at step 0
        _ids = torch.randint(5, VOCAB_SIZE, (2, 40), device=device)
        a = backbone_hidden(model, _ids); b, _ = lm.backbone(_ids)
    QSTATS["hidden_diff_vs_fp"] = (a - b).abs().max().item()
    print(f"final hidden states, NF4 model vs full precision: max |diff| {QSTATS['hidden_diff_vs_fp']:.3e} (quantisation, not a bug)")
    d = lm_cfg.d_model
    model.head = nn.Linear(d, d, bias=False).to(device) if USE_HEAD else None
    if USE_HEAD:
        with torch.no_grad(): model.head.weight.copy_(torch.eye(d))
    tot_fp = sum(p.numel() for p in lm.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_lora = sum(p.numel() for nm, p in model.named_parameters() if p.requires_grad and not nm.startswith("head."))
    print(f"params: Phase-1 model {tot_fp/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot_fp:.2f}%) = LoRA {n_lora/1e6:.3f}M "
          f"(r={hp['r']}, alpha={hp['alpha']}) + head {(tr - n_lora)/1e6:.3f}M")
    model.n_total, model.n_trainable, model.n_peft = tot_fp, tr, n_lora
    return model
QSTATS = {}

def backbone_hidden(model, ids, mask=None):
    # Backbone.forward for right-padded chunks (causal: no mask needed; NoPE: rope is None), with activation checkpointing.
    bb = model.backbone; N, T = ids.shape
    x = bb.tok_emb(ids); rope_cs = None
    if bb.rope is not None or bb.pos_emb is not None:
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(N, T)
        if bb.rope is not None: rope_cs = bb.rope.tables(pos)
        else: x = x + bb.pos_emb(pos)
    x = bb.emb_drop(x)
    for blk in bb.blocks:
        if model.training and GRAD_CKPT and torch.is_grad_enabled():
            x = checkpoint(lambda t, blk=blk: blk(t, attn_bias=None, rope_cs=rope_cs)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x, attn_bias=None, rope_cs=rope_cs)
    return bb.norm_f(x)

with torch.no_grad():
    _ids = torch.randint(5, VOCAB_SIZE, (2, 40), device=device)
    _a = backbone_hidden(lm, _ids); _b, _ = lm.backbone(_ids)
assert (_a - _b).abs().max().item() < 1e-4, "backbone_hidden differs from the Phase-1 forward"
# NF4 round-trip unit test on a random normal matrix: error should be ~0.1 (4 bits on Gaussian data)
_W = torch.randn(96, 200); _q = nf4_quantize(_W); _Wq = nf4_dequantize(_q, NF4)
print(f"NF4 unit test on N(0,1) data: relative error {((_W - _Wq).norm() / _W.norm()).item():.4f}; codes in [0,15]: "
      f"{int((_q['packed'] & 15).max())},{int((_q['packed'] >> 4).max())}")

NF4 levels: [-1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.091, 0.0, 0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.723, 1.0]
NF4 unit test on N(0,1) data: relative error 0.0924; codes in [0,15]: 15,15


In [16]:
# Cell 16 — Pooling, chunk embedding, document scoring (mean / maxsim), evaluation, fusion
def make_batch(seqs):
    # seqs: list of uint16 arrays (chunk bodies). Returns ids [N, T] right-padded with [BOS] body [EOS], lengths [N].
    lens = [len(s) + 2 for s in seqs]; T = max(lens)
    ids = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, 0] = BOS_ID; ids[i, 1:1 + len(s)] = torch.from_numpy(s.astype(np.int64)); ids[i, 1 + len(s)] = EOS_ID
    return ids.to(device, non_blocking=True), torch.tensor(lens, device=device)

def pool_hidden(h, lens, how):
    # h [N, T, d] (right padding), lens [N] -> [N, d]
    N, T, _ = h.shape
    t = torch.arange(T, device=h.device).unsqueeze(0)
    m = (t < lens.unsqueeze(1)).to(h.dtype)
    if how == "mean":  w = m
    elif how == "wmean": w = m * (t + 1).to(h.dtype)                 # SGPT position weights
    elif how == "last":  return h[torch.arange(N, device=h.device), lens - 1]
    else: raise ValueError(how)
    return (h * w.unsqueeze(-1)).sum(1) / w.sum(1, keepdim=True)

def encode(model, ids, lens, poolings):
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h = backbone_hidden(model, ids)
    out = {}
    for p in poolings:
        e = pool_hidden(h.float(), lens, p)
        if getattr(model, "head", None) is not None: e = model.head(e)
        out[p] = F.normalize(e, dim=-1)
    return out

@torch.no_grad()
def embed_keys(model, keys, chunk_fn, poolings):
    # -> {pooling: [n_chunks, d] float32 (GPU)}, owner [n_chunks] (index into keys), sorted by owner
    model.eval()
    seqs, owner = [], []
    for i, k in enumerate(keys):
        for c in chunk_fn(k): seqs.append(c); owner.append(i)
    order = np.argsort([len(s) for s in seqs], kind="stable")
    E = {p: torch.empty(len(seqs), lm_cfg.d_model, device=device) for p in poolings}
    for b in range(0, len(order), EMB_BATCH):
        idx = order[b:b + EMB_BATCH]
        ids, lens = make_batch([seqs[j] for j in idx])
        out = encode(model, ids, lens, poolings)
        for p in poolings: E[p][torch.as_tensor(idx, device=device)] = out[p]
    return E, torch.tensor(owner, device=device)

def embed_split(model, split, poolings, q_subset=None):
    qs = QRY[split] if q_subset is None else QRY[split][:q_subset]
    t0 = time.time()
    Eq, oq = embed_keys(model, [q["key"] for q in qs], query_chunks, poolings)
    Ed, od = embed_keys(model, POOL[split], doc_chunks, poolings)
    return dict(Eq=Eq, oq=oq, Ed=Ed, od=od, nq=len(qs), nd=len(POOL[split]), secs=time.time() - t0, n_chunks=len(od) + len(oq))

def seg_mean(E, owner, n):
    s = torch.zeros(n, E.shape[1], device=E.device).index_add_(0, owner, E)
    return F.normalize(s, dim=-1)

@torch.no_grad()
def dense_scores(emb, pooling, mode, center=False, block=256):
    Eq, Ed, oq, od = emb["Eq"][pooling], emb["Ed"][pooling], emb["oq"], emb["od"]
    if center:
        mu = Ed.mean(0, keepdim=True); Eq = F.normalize(Eq - mu, dim=-1); Ed = F.normalize(Ed - mu, dim=-1)
    nq, nd = emb["nq"], emb["nd"]
    if mode == "mean":
        S = seg_mean(Eq, oq, nq) @ seg_mean(Ed, od, nd).T
    else:                                               # maxsim: mean over query chunks of the max over candidate chunks
        S = torch.zeros(nq, nd, device=device); cnt = torch.zeros(nq, device=device)
        for b in range(0, len(oq), block):
            sim = Eq[b:b + block] @ Ed.T                                        # [qb, n_doc_chunks]
            mx = torch.full((sim.shape[0], nd), -2.0, device=device).scatter_reduce(1, od.expand(sim.shape[0], -1), sim, "amax")
            S.index_add_(0, oq[b:b + block], mx); cnt.index_add_(0, oq[b:b + block], torch.ones(len(mx), device=device))
        S = S / cnt.clamp(min=1).unsqueeze(1)
    return S.float().cpu().numpy()

def self_mask(S, split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; S = S.copy()
    for i, q in enumerate(QRY[split][: S.shape[0]]):
        if q["key"] in col and not (q["gold"] and q["key"] in q["gold"]): S[i, col[q["key"]]] = -np.inf
    return S

def bm25_matrix(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    S = np.zeros((len(qs), len(POOL[split])), dtype=np.float32)
    for i, q in enumerate(qs):
        lst = BM25[split][q["key"]]; lo = min(v for _, v in lst) if lst else 0.0
        S[i, :] = lo - 1.0
        for c, v in lst:
            if c in col: S[i, col[c]] = v
    return S

def topk(S, k):
    k = min(k, S.shape[1]); idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(S, idx, 1), axis=1); return np.take_along_axis(idx, o, 1)
def gold_cols(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    return [set(col[g] for g in q["gold"]) for q in qs]
def evaluate(S, split, ks=(K_EVAL,), recall_ks=RECALL_KS):
    gc_ = gold_cols(split, S.shape[0]); N = len(gc_); G = sum(len(g) for g in gc_)
    T = topk(S, max(max(ks), max(recall_ks))); out = {}
    for k in ks:
        H = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N))
        out[f"H@{k}"] = H; out[f"P@{k}"] = H / (k * N); out[f"R@{k}"] = H / max(G, 1); out[f"F1@{k}"] = 2 * H / (k * N + G)
    for k in recall_ks: out[f"recall@{k}"] = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N)) / max(G, 1)
    rr = [next((1 / (j + 1) for j, c in enumerate(T[i].tolist()) if c in gc_[i]), 0.0) for i in range(N)]
    out["MRR"] = float(np.mean(rr)); return out

def znorm(S):
    S = S.astype(np.float64); f = np.isfinite(S)
    m = np.where(f, S, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    v = np.where(f, (S - m) ** 2, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    return np.where(f, (S - m) / np.sqrt(v + 1e-12), -np.inf)
def rrf(*Ss):
    out = np.zeros_like(Ss[0], dtype=np.float64)
    for S in Ss:
        r = np.empty_like(S, dtype=np.int64); o = np.argsort(-S, axis=1); r[np.arange(len(S))[:, None], o] = np.arange(S.shape[1])
        out += 1.0 / (RRF_K + 1 + r)
    return out
def fuse_search(Sd, Sb, split):
    # Sd, Sb already self-masked. Returns best (alpha, eval) by H@7 (ties: recall@100), and the whole curve.
    zd, zb = znorm(Sd), znorm(Sb); curve = []
    for a in ALPHA_GRID:
        e = evaluate(a * np.nan_to_num(zd, neginf=-1e9) + (1 - a) * np.nan_to_num(zb, neginf=-1e9), split)
        curve.append((float(a), e["H@7"], e["F1@7"], e["recall@100"]))
    best = max(curve, key=lambda t: (t[1], t[3], -abs(t[0] - 0.5)))
    return best, curve
def fused(Sd, Sb, alpha):
    zd, zb = znorm(Sd), znorm(Sb)
    return alpha * np.nan_to_num(zd, neginf=-1e9) + (1 - alpha) * np.nan_to_num(zb, neginf=-1e9)

## Zero-shot transfer of the Phase-1 representations (before any training)
One pass over dev produces all three poolings (they share the hidden states). For each pooling and score mode we report **dense-only** dev metrics, the **best fusion with BM25**, and the effect of **centering** (subtracting the pool's mean chunk vector before the cosine). Centering targets *anisotropy*: language-model hidden states share a large common direction, so all cosines are close to each other and dominated by it (Ethayarajh 2019); removing the mean exposes the directions that actually differ between documents. If centering helps a lot zero-shot, that is direct evidence of anisotropy in the Phase-1 model.

In [17]:
# Cell 17 — Build the model; BM25-only dev reference; zero-shot table; choose POOLING
seed_everything(SEED)
model = build_model()
DEV_Q = SMOKE_DEV_Q if SMOKE_TEST else None
Sb_dev = self_mask(bm25_matrix("dev", DEV_Q), "dev")
e_b = evaluate(Sb_dev, "dev")
print(f"BM25 only (dev{'[:%d]' % DEV_Q if DEV_Q else ''}): H@7 {e_b['H@7']} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")

POOLS_ALL = ("mean", "wmean", "last")
emb0 = embed_split(model, "dev", POOLS_ALL, DEV_Q)
print(f"dev embedded: {emb0['n_chunks']} chunks in {emb0['secs']:.0f}s")
ZS = []
for p in POOLS_ALL:
    for mode in ("mean", "maxsim"):
        for center in (False, True):
            Sd = self_mask(dense_scores(emb0, p, mode, center), "dev")
            e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
            ZS.append(dict(pooling=p, mode=mode, center=center, H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], MRR=e["MRR"],
                           fused_H7=H, fused_F1=f1, alpha=a))
            print(f"  zero-shot {p:5s} {mode:6s} center={center!s:5s}: dense H@7 {e['H@7']:4d} F1 {e['F1@7']:.4f} r@100 {e['recall@100']:.3f} | "
                  f"fused H@7 {H:4d} F1 {f1:.4f} (alpha {a})")
if POOLING == "auto":
    POOLING = max(POOLS_ALL, key=lambda p: max(r["H7"] for r in ZS if r["pooling"] == p))
print("POOLING ->", POOLING)
del emb0; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None

NF4: 84 Linear layers quantised in 2s | relative error ||W-Q(W)||/||W||: mean 0.0926 max 0.1015 | weights 73.5 MB (fp16) -> 19.0 MB (4.127 bits/weight incl. scales)
LM loss on a legal sentence: full precision 2.0145 | NF4 base 1.9726
final hidden states, NF4 model vs full precision: max |diff| 2.561e+00 (quantisation, not a bug)
params: Phase-1 model 54.93M | trainable 2.154M (3.92%) = LoRA 1.892M (r=16, alpha=32) + head 0.262M
BM25 only (dev): H@7 395 F1@7 0.4391 recall@100 0.754 MRR 0.895
dev embedded: 11164 chunks in 65s
  zero-shot mean  mean   center=False: dense H@7  201 F1 0.2235 r@100 0.619 | fused H@7  396 F1 0.4402 (alpha 0.4)
  zero-shot mean  mean   center=True : dense H@7  213 F1 0.2368 r@100 0.627 | fused H@7  396 F1 0.4402 (alpha 0.3)
  zero-shot mean  maxsim center=False: dense H@7  205 F1 0.2279 r@100 0.605 | fused H@7  396 F1 0.4402 (alpha 0.1)
  zero-shot mean  maxsim center=True : dense H@7  221 F1 0.2457 r@100 0.602 | fused H@7  397 F1 0.4414 (alpha 0.15)
  zero-sh

## Training
Each step: `BATCH_PAIRS` (query, positive) pairs from a shuffled list of all train gold pairs (an epoch visits every gold pair once, so queries with many citations get proportionally more steps — consistent with the micro-averaged metric). For each pair one hard negative is drawn from the query's BM25 top-`HN_TOP` minus its gold and itself. Every document contributes `C_TRAIN` random chunks and every query `QC_TRAIN` random evidence-view chunks (fresh random choices every step = augmentation). Embeddings are the normalised means of the chunk vectors. The forward runs in pieces of `FWD_MICRO` sequences with activation checkpointing; the loss needs all `2B` documents together, so the backward is one call.

In [18]:
# Cell 18 — Training loop (multi-positive InfoNCE with false-negative masking), per-epoch dev evaluation
TRQ = [q for q in QRY["train"] if q["gold"]]
TR_PAIRS = [(qi, g) for qi, q in enumerate(TRQ) for g in q["gold"]]
TR_GOLD = [set(q["gold"]) for q in TRQ]
TR_HN = [[c for c, _ in BM25["train"][q["key"]][:HN_TOP] if c not in TR_GOLD[qi] and c != q["key"]] for qi, q in enumerate(TRQ)]
print(f"train: {len(TRQ)} queries with gold | {len(TR_PAIRS)} (query, positive) pairs | queries without a hard negative: {sum(not h for h in TR_HN)}")
_rank = [{c: r + 1 for r, (c, _) in enumerate(BM25["train"][q["key"]])} for q in TRQ]
TR_POS_RANK = [_rank[qi].get(g, 10**6) for qi, g in TR_PAIRS]
TR_W = [1.0 + (HARD_BOOST if r > HARD_RANK else 0.0) for r in TR_POS_RANK]
_hard = np.mean([r > HARD_RANK for r in TR_POS_RANK])
print(f"train positives ranked by BM25: top-7 {np.mean([r <= 7 for r in TR_POS_RANK]):.1%} | beyond {HARD_RANK}: {_hard:.1%} | "
      f"beyond the saved top-300: {np.mean([r > 300 for r in TR_POS_RANK]):.1%} | POS_SAMPLING={POS_SAMPLING}"
      + (f" -> hard positives become {_hard * (1 + HARD_BOOST) / (1 + _hard * HARD_BOOST):.1%} of each epoch" if POS_SAMPLING == "bm25_hard" else ""))

def sample_chunks(ch, n, rng): return ch if len(ch) <= n else [ch[i] for i in sorted(rng.sample(range(len(ch)), n))]

def step_loss(model, batch, rng):
    # batch: list of (qi, pos_key). Returns loss, stats.
    docs = []
    for qi, pk in batch:
        docs.append(pk)
        hn = TR_HN[qi]; docs.append(rng.choice(hn) if hn else rng.choice(POOL["train"]))
    seqs, owner = [], []
    for i, (qi, _) in enumerate(batch):
        for c in sample_chunks(query_chunks(TRQ[qi]["key"]), QC_TRAIN, rng): seqs.append(c); owner.append(i)
    nQ = len(batch)
    for j, k in enumerate(docs):
        for c in sample_chunks(doc_chunks(k), C_TRAIN, rng): seqs.append(c); owner.append(nQ + j)
    embs = []
    for b in range(0, len(seqs), FWD_MICRO):
        ids, lens = make_batch(seqs[b:b + FWD_MICRO]); embs.append(encode(model, ids, lens, (POOLING,))[POOLING])
    E = torch.cat(embs); own = torch.tensor(owner, device=device)
    V = seg_mean(E, own, nQ + len(docs)); Q, D = V[:nQ], V[nQ:]
    logits = (Q @ D.T) / TAU                                                  # [B, 2B]
    mask = torch.zeros_like(logits, dtype=torch.bool)
    for i, (qi, _) in enumerate(batch):
        for j, k in enumerate(docs):
            if j != 2 * i and (k in TR_GOLD[qi] or k == TRQ[qi]["key"]): mask[i, j] = True
    logits = logits.masked_fill(mask, float("-inf"))
    target = torch.arange(nQ, device=device) * 2
    loss = F.cross_entropy(logits.float(), target)
    with torch.no_grad():
        acc = (logits.argmax(1) == target).float().mean().item()
        gap = (logits[torch.arange(nQ), target] - logits[torch.arange(nQ), target + 1]).mean().item() * TAU   # cos(q,pos) - cos(q,hard neg)
    return loss, dict(acc=acc, gap=gap, masked=int(mask.sum().item()), seqs=len(seqs))

def param_groups(model):
    # Full FT: decoupled weight decay on 2-D matrices only (not the embedding, norm gains); the head at head_lr without decay.
    hp = HP[METHOD]; dec, nodec, head = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith("head."): head.append(p)
        elif p.ndim >= 2 and "tok_emb" not in n and "pos_emb" not in n: dec.append(p)
        else: nodec.append(p)
    g = [dict(params=dec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]),
         dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0)]
    if head: g.append(dict(params=head, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0))
    print(f"param groups: decay {sum(p.numel() for p in dec)/1e6:.2f}M | no-decay {sum(p.numel() for p in nodec)/1e6:.2f}M | head {sum(p.numel() for p in head)/1e6:.3f}M")
    return g

def dev_eval(model, emb=None):
    emb = emb or embed_split(model, "dev", (POOLING,), DEV_Q)
    out = {}
    for mode in ("mean", "maxsim"):
        Sd = self_mask(dense_scores(emb, POOLING, mode), "dev")
        e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
        out[mode] = dict(H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], fused_H7=H, fused_F1=f1, alpha=a)
    return out, emb

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    spe = math.ceil(len(TR_PAIRS) / BATCH_PAIRS); total = epochs * spe
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"{spe} steps/epoch | total {total} steps | warmup {warm}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, step, done, t_train = [], 0, False, time.time()
    best = dict(score=None, epoch=0, state=None)
    for ep in range(1, epochs + 1):
        rng = random.Random(SEED + ep)
        if POS_SAMPLING == "bm25_hard": pairs = rng.choices(TR_PAIRS, weights=TR_W, k=len(TR_PAIRS))
        else: pairs = TR_PAIRS[:]; rng.shuffle(pairs)
        model.train(); t0 = time.time(); L, A, Gp = [], [], []
        for b in range(0, len(pairs), BATCH_PAIRS):
            loss, st = step_loss(model, pairs[b:b + BATCH_PAIRS], rng)
            m = lr_mult(step)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * m
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
            L.append(loss.item()); A.append(st["acc"]); Gp.append(st["gap"])
            if step == 10:
                per = (time.time() - t0) / 10
                print(f"  >> speed {per:.2f} s/step ({st['seqs']} seqs) | projected {per * spe / 60:.1f} min/epoch"
                      + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
            if step % 50 == 0:
                print(f"  ep {ep} step {step}/{total} | loss {np.mean(L[-50:]):.4f} | in-batch acc {np.mean(A[-50:]):.3f} | "
                      f"cos(pos)-cos(hn) {np.mean(Gp[-50:]):+.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
            if max_steps and step >= max_steps: done = True; break
            if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        ev, _ = dev_eval(model)
        score = max(ev[m]["fused_H7"] for m in ev) + 1e-3 * max(ev[m]["H7"] for m in ev)
        rec = dict(epoch=ep, loss=float(np.mean(L)), acc=float(np.mean(A)), gap=float(np.mean(Gp)), steps=step, secs=time.time() - t0, dev=ev)
        history.append(rec)
        print(f"epoch {ep}: loss {rec['loss']:.4f} acc {rec['acc']:.3f} | dev dense H@7 mean {ev['mean']['H7']} / maxsim {ev['maxsim']['H7']} | "
              f"fused H@7 mean {ev['mean']['fused_H7']} (a={ev['mean']['alpha']}) / maxsim {ev['maxsim']['fused_H7']} (a={ev['maxsim']['alpha']})")
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0)
    return history, best, stats

history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)
print(f"\nbest epoch {best['epoch']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

train: 817 queries with gold | 5308 (query, positive) pairs | queries without a hard negative: 0
train positives ranked by BM25: top-7 36.6% | beyond 50: 30.7% | beyond the saved top-300: 15.2% | POS_SAMPLING=uniform
param groups: decay 1.89M | no-decay 0.00M | head 0.262M
166 steps/epoch | total 664 steps | warmup 39
  >> speed 7.97 s/step (303 seqs) | projected 22.1 min/epoch | peak GPU 4.9 GB
  ep 1 step 50/664 | loss 2.7873 | in-batch acc 0.297 | cos(pos)-cos(hn) -0.0148 | lr x0.999 | 398s
  ep 1 step 100/664 | loss 2.4215 | in-batch acc 0.364 | cos(pos)-cos(hn) -0.0032 | lr x0.979 | 796s
  ep 1 step 150/664 | loss 2.4368 | in-batch acc 0.369 | cos(pos)-cos(hn) -0.0035 | lr x0.929 | 1194s
epoch 1: loss 2.5355 acc 0.346 | dev dense H@7 mean 296 / maxsim 291 | fused H@7 mean 397 (a=0.4) / maxsim 399 (a=0.3)
  ep 2 step 200/664 | loss 2.2633 | in-batch acc 0.378 | cos(pos)-cos(hn) +0.0048 | lr x0.854 | 271s
  ep 2 step 250/664 | loss 2.1855 | in-batch acc 0.416 | cos(pos)-cos(hn) +0.0

## Final dev evaluation and analysis
With the best epoch restored: dense-only and fused scores for both score modes (the better one becomes `SCORE_MODE`), the full `α` curve, reciprocal-rank fusion, the F1@K curve, the rank of every gold case under the fused score (versus under BM25 alone), how many gold cases the fused ranking **adds** to BM25's top 7 and how many it **loses**, and the per-query hit distribution.

In [19]:
# Cell 19 — Final dev evaluation: score mode, alpha curve, RRF, F1@K, gold ranks, gains vs BM25
ev_final, emb_dev = dev_eval(model)
if SCORE_MODE == "auto":
    SCORE_MODE = max(("mean", "maxsim"), key=lambda m: (ev_final[m]["fused_H7"], ev_final[m]["H7"]))
Sd_dev = self_mask(dense_scores(emb_dev, POOLING, SCORE_MODE), "dev")
(ALPHA, H_f, F1_f, r100_f), curve = fuse_search(Sd_dev, Sb_dev, "dev")
Sf_dev = fused(Sd_dev, Sb_dev, ALPHA)
e_d, e_f = evaluate(Sd_dev, "dev"), evaluate(Sf_dev, "dev", ks=tuple(range(1, 21)))
e_rrf = evaluate(rrf(Sd_dev, Sb_dev), "dev")
print(f"SCORE_MODE={SCORE_MODE} | POOLING={POOLING} | ALPHA={ALPHA}")
print(f"  BM25 only : H@7 {e_b['H@7']:4d} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")
print(f"  dense only: H@7 {e_d['H@7']:4d} F1@7 {e_d['F1@7']:.4f} recall@100 {e_d['recall@100']:.3f} MRR {e_d['MRR']:.3f}")
print(f"  fused     : H@7 {e_f['H@7']:4d} F1@7 {e_f['F1@7']:.4f} recall@100 {e_f['recall@100']:.3f} MRR {e_f['MRR']:.3f}  (P@7 {e_f['P@7']:.3f} R@7 {e_f['R@7']:.3f})")
print(f"  RRF       : H@7 {e_rrf['H@7']:4d} F1@7 {e_rrf['F1@7']:.4f} recall@100 {e_rrf['recall@100']:.3f}")
print("  alpha curve (alpha, H@7):", [(a, H) for a, H, _, _ in curve])
print("  F1@K (fused):", {k: round(e_f[f'F1@{k}'], 4) for k in range(1, 21)})

gcols = gold_cols("dev", Sd_dev.shape[0])
def ranks(S):
    o = np.argsort(-S, axis=1); r = np.empty_like(o); r[np.arange(len(o))[:, None], o] = np.arange(o.shape[1])
    return np.asarray([r[i, c] + 1 for i, g in enumerate(gcols) for c in g])
rb, rf, rd = ranks(Sb_dev), ranks(Sf_dev), ranks(Sd_dev)
print(f"  gold rank median: BM25 {np.median(rb):.0f} | dense {np.median(rd):.0f} | fused {np.median(rf):.0f}  "
      f"(top-100 share: {np.mean(rb <= 100):.3f} / {np.mean(rd <= 100):.3f} / {np.mean(rf <= 100):.3f})")
Tb, Tf = topk(Sb_dev, 7), topk(Sf_dev, 7)
gain = sum(len((set(Tf[i].tolist()) - set(Tb[i].tolist())) & gcols[i]) for i in range(len(gcols)))
loss_ = sum(len((set(Tb[i].tolist()) - set(Tf[i].tolist())) & gcols[i]) for i in range(len(gcols)))
hits = np.asarray([len(set(Tf[i].tolist()) & gcols[i]) for i in range(len(gcols))])
print(f"  vs BM25 top-7: gold cases gained {gain} | lost {loss_} | net {gain - loss_:+d}")
print(f"  hits@7 per query (fused): {dict(sorted(Counter(hits.tolist()).items()))} | zero-hit queries {(hits == 0).mean():.1%}")
FINAL = dict(score_mode=SCORE_MODE, pooling=POOLING, alpha=ALPHA, bm25=e_b, dense=e_d,
             fused={k: v for k, v in e_f.items() if k.endswith("@7") or k.startswith("recall") or k == "MRR"},
             f1_curve={k: e_f[f"F1@{k}"] for k in range(1, 21)}, rrf=e_rrf, alpha_curve=curve,
             gold_rank_median=dict(bm25=float(np.median(rb)), dense=float(np.median(rd)), fused=float(np.median(rf))),
             gained=gain, lost=loss_, zero_hit=float((hits == 0).mean()), by_epoch_modes=ev_final)

SCORE_MODE=mean | POOLING=wmean | ALPHA=0.7
  BM25 only : H@7  395 F1@7 0.4391 recall@100 0.754 MRR 0.895
  dense only: H@7  302 F1@7 0.3357 recall@100 0.717 MRR 0.729
  fused     : H@7  402 F1@7 0.4469 recall@100 0.786 MRR 0.900  (P@7 0.487 R@7 0.413)
  RRF       : H@7  371 F1@7 0.4125 recall@100 0.783
  alpha curve (alpha, H@7): [(0.0, 395), (0.05, 397), (0.1, 397), (0.15, 397), (0.2, 397), (0.25, 395), (0.3, 396), (0.35, 396), (0.4, 396), (0.45, 395), (0.5, 394), (0.55, 397), (0.6, 395), (0.65, 398), (0.7, 402), (0.75, 399), (0.8, 391), (0.85, 384), (0.9, 373), (0.95, 343), (1.0, 302)]
  F1@K (fused): {1: 0.1852, 2: 0.2928, 3: 0.3617, 4: 0.3986, 5: 0.4248, 6: 0.4402, 7: 0.4469, 8: 0.4413, 9: 0.4373, 10: 0.431, 11: 0.4218, 12: 0.4102, 13: 0.4021, 14: 0.3909, 15: 0.3821, 16: 0.3719, 17: 0.3619, 18: 0.3532, 19: 0.3446, 20: 0.3372}
  gold rank median: BM25 12 | dense 20 | fused 12  (top-100 share: 0.754 / 0.717 / 0.786)
  vs BM25 top-7: gold cases gained 26 | lost 19 | net +7
  hits@7 p

In [20]:
# Cell 20 — Test: embed, fuse with BM25, prediction.json -> zip (full runs only)
def validate_prediction(pred, split="test"):
    pool_ids = {DOCS[k]["id"] for k in POOL[split]}; qids = [q["id"] for q in QRY[split]]
    assert set(pred) == set(qids), f"missing/extra queries: {len(set(qids) - set(pred))} missing, {len(set(pred) - set(qids))} extra"
    for qid, lst in pred.items():
        assert isinstance(lst, list) and len(lst) == K_EVAL and len(set(lst)) == K_EVAL, f"{qid}: need {K_EVAL} unique ids"
        assert all(isinstance(c, str) and c in pool_ids for c in lst), f"{qid}: id not in the {split} pool"
    return True

TEST = {}
if not SMOKE_TEST:
    t0 = time.time()
    emb_test = embed_split(model, "test", (POOLING,))
    Sd_test = self_mask(dense_scores(emb_test, POOLING, SCORE_MODE), "test")
    Sb_test = self_mask(bm25_matrix("test"), "test")
    Sf_test = fused(Sd_test, Sb_test, ALPHA)
    T7 = topk(Sf_test, K_EVAL)
    PRED = {q["id"]: [DOCS[POOL["test"][j]]["id"] for j in T7[i].tolist()] for i, q in enumerate(QRY["test"])}
    validate_prediction(PRED)
    sub_dir = RUN_DIR / "submission"; sub_dir.mkdir(exist_ok=True)
    (sub_dir / "prediction.json").write_text(json.dumps(PRED, indent=1))
    zpath = RUN_DIR / f"submission_{RUN_NAME}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_dir / "prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z: assert z.namelist() == ["prediction.json"]
    Tb7 = topk(Sb_test, K_EVAL)
    overlap = np.mean([len(set(T7[i].tolist()) & set(Tb7[i].tolist())) / K_EVAL for i in range(len(T7))])
    TEST = dict(seconds=time.time() - t0, embed_seconds=emb_test["secs"], chunks=emb_test["n_chunks"], top7_overlap_with_bm25=float(overlap))
    print(f"test: {len(PRED)} queries -> {zpath} | embed {emb_test['secs']:.0f}s ({emb_test['n_chunks']} chunks) | "
          f"top-7 overlap with BM25 {overlap:.2f}")
    if SAVE_SCORES:
        def top_list(S, split):
            T = topk(S, SAVE_TOP)
            return {q["id"]: [[DOCS[POOL[split][j]]["id"], round(float(S[i, j]), 5)] for j in T[i].tolist() if np.isfinite(S[i, j])]
                    for i, q in enumerate(QRY[split][: S.shape[0]])}
        (RUN_DIR / "dense_scores.json").write_text(json.dumps(dict(run=RUN_NAME, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA,
                                                                   dev=top_list(Sd_dev, "dev"), test=top_list(Sd_test, "test"))))
        print("saved dense top-%d scores for dev/test" % SAVE_TOP)

test: 237 queries -> /kaggle/working/runs/p7_dec01_qlora/submission_p7_dec01_qlora.zip | embed 112s (18838 chunks) | top-7 overlap with BM25 0.88
saved dense top-300 scores for dev/test


## KD teacher export (for the distillation run)
For every train query: the teacher's **raw cosine scores** (chosen pooling and score mode) over its BM25 top-300 ∪ its gold set, keyed by the pool's ID strings, together with `τ`. The student run turns them into a listwise target `softmax(s/τ)` over the same candidate list (and pairwise margins), so it learns the teacher's *ranking*, not just the binary gold labels. Only cosine similarities are exported, so the student can be any architecture with any embedding size.

In [21]:
# Cell 21 — Save trainable weights + log; teacher export
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    if isinstance(o, Path): return str(o)
    return o
torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, RUN_DIR / "trainable_final.pt")
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], qstats=QSTATS, params_peft=model.n_peft, pos_sampling=POS_SAMPLING, hard=(HARD_RANK, HARD_BOOST), view=dict(chunk=CHUNK_TOKENS, max_doc_chunks=MAX_DOC_CHUNKS, query_view=QUERY_VIEW,
           ctx_window=CTX_WINDOW, q_max_chunks=Q_MAX_CHUNKS), train=dict(epochs=EPOCHS, B=BATCH_PAIRS, C=C_TRAIN, QC=QC_TRAIN, hn_top=HN_TOP, tau=TAU),
           params_total=model.n_total, params_trainable=model.n_trainable, zero_shot=ZS, history=history, best_epoch=best["epoch"],
           stats=stats, final=FINAL, test=TEST, bm25_cfg=BM25_CFG)
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=1))

TEACHER = {}
if EXPORT_TEACHER and not SMOKE_TEST:
    t0 = time.time()
    emb_tr = embed_split(model, "train", (POOLING,))
    S_tr = self_mask(dense_scores(emb_tr, POOLING, SCORE_MODE), "train")
    col = {k: j for j, k in enumerate(POOL["train"])}
    exp = {}
    for i, q in enumerate(QRY["train"]):
        cands = [c for c, _ in BM25["train"][q["key"]]] + [g for g in (q["gold"] or [])]
        seen, row = set(), []
        for c in cands:
            if c in seen or c == q["key"] or c not in col: continue
            seen.add(c); row.append([DOCS[c]["id"], round(float(S_tr[i, col[c]]), 5)])
        exp[q["id"]] = row
    tpath = RUN_DIR / f"teacher_{RUN_NAME}.json"
    tpath.write_text(json.dumps(dict(run=RUN_NAME, tau=TAU, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA, scores=exp)))
    TEACHER = dict(seconds=time.time() - t0, queries=len(exp), mean_list=float(np.mean([len(v) for v in exp.values()])),
                   mb=tpath.stat().st_size / 1e6)
    print(f"teacher export: {TEACHER} -> {tpath}")
for f in sorted(RUN_DIR.iterdir()): print(f"   {f.name:40s} {f.stat().st_size / 1e6 if f.is_file() else 0:8.2f} MB")

teacher export: {'seconds': 273.5675172805786, 'queries': 827, 'mean_list': 300.9782345828295, 'mb': 6.208332} -> /kaggle/working/runs/p7_dec01_qlora/teacher_p7_dec01_qlora.json
   best_trainable.pt                            8.67 MB
   dense_scores.json                            2.66 MB
   run_log.json                                 0.01 MB
   submission                                   0.00 MB
   submission_p7_dec01_qlora.zip                0.01 MB
   teacher_p7_dec01_qlora.json                  6.21 MB
   trainable_final.pt                           8.67 MB


In [22]:
# Cell 22 — PASTE TO CLAUDE
def _r(o, n=4):
    if isinstance(o, dict): return {k: _r(v, n) for k, v in o.items()}
    if isinstance(o, list): return [_r(v, n) for v in o]
    if isinstance(o, float): return float(f"{o:.3g}") if 0 < abs(o) < 1e-2 else round(o, n)
    return o
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], qstats=QSTATS, params_peft=model.n_peft, pos_sampling=POS_SAMPLING, params_trainable=model.n_trainable, params_total=model.n_total,
             zero_shot=ZS, history=[{k: v for k, v in h.items()} for h in history], best_epoch=best["epoch"], stats=stats,
             final={k: v for k, v in FINAL.items() if k not in ("alpha_curve",)}, alpha_curve=[(a, H) for a, H, _, _ in FINAL["alpha_curve"]],
             test=TEST, teacher=TEACHER)
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_r(_jsonable(short)), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p7_dec01_qlora","smoke":false,"hp":{"lr":0.0003,"head_lr":0.00025,"wd":0.0,"r":16,"alpha":32,"dropout":0.05},"qstats":{"n_layers":84,"rel_err_mean":0.0926,"rel_err_max":0.1015,"mb_fp16":73.5,"mb_nf4":18.9585,"bits_per_weight":4.127,"seconds":2.316,"lm_loss_fp":2.0145,"lm_loss_nf4":1.9726,"hidden_diff_vs_fp":2.5611},"params_peft":1892352,"pos_sampling":"uniform","params_trainable":2154496,"params_total":54931968,"zero_shot":[{"pooling":"mean","mode":"mean","center":false,"H7":201,"F1":0.2235,"r100":0.6187,"MRR":0.5282,"fused_H7":396,"fused_F1":0.4402,"alpha":0.4},{"pooling":"mean","mode":"mean","center":true,"H7":213,"F1":0.2368,"r100":0.6269,"MRR":0.5464,"fused_H7":396,"fused_F1":0.4402,"alpha":0.3},{"pooling":"mean","mode":"maxsim","center":false,"H7":205,"F1":0.2279,"r100":0.6053,"MRR":0.5977,"fused_H7":396,"fused_F1":0.4402,"alpha":0.1},{"pooling":"mean","mode":"maxsim","center":true,"H7":221,"F1":0.2457,"r100":0.6023,"MRR":0